# BERT'ni sentiment tahlil uchun fine-tuning qilish

Oldingi darslarda modellarni noldan o'qitgan edik: RNN, LSTM, GRU va word2vec — hammasi
tasodifiy og'irliklardan boshlangan. BERT bilan yondashuv boshqacha.

**BERT** (Bidirectional Encoder Representations from Transformers) — Google 2018-yilda
chiqargan Transformer *encoder* modeli. U Vikipediya va kitoblar korpusida (~3.3 milliard so'z)
ikkita vazifa bilan oldindan o'qitilgan:

- **Masked Language Modeling** — jumladagi 15% so'z `[MASK]` bilan yashiriladi, model ularni
  ikki tomondagi kontekstga qarab tiklaydi. Aynan shu "ikki tomonlama" qarash BERT'ni
  GPT kabi chapdan-o'ngga o'qiydigan modellardan farqlaydi.
- **Next Sentence Prediction** — ikkita jumla ketma-ket kelganmi yoki yo'qmi.

Natijada BERT tilning grammatikasi va ma'nosini allaqachon "biladi". Bizga esa faqat
sentiment tahlil qilish qoladi — bu **fine-tuning** deb ataladi:

1. Oldindan o'qitilgan encoder'ni olamiz (110M parametr, tayyor bilim).
2. Uning tepasiga kichkina, tasodifiy boshlang'ich qiymatli **klassifikatsiya boshi** qo'yamiz.
3. Butun tarmoqni **juda kichik** learning rate bilan bir-ikki epoxa o'qitamiz.

Datasat sifatida **IMDb Large Movie Review** to'plamini ishlatamiz — Stanford tayyorlagan
50 000 ta film sharhi (25k train + 25k test), yarmi ijobiy, yarmi salbiy. Bu sentiment
tahlil bo'yicha eng mashhur benchmark.

## Kerakli kutubxonalar

`transformers` — BERT modeli va uning tokenizatori, `datasets` — IMDb to'plamini yuklab olish,
`sklearn` — metrikalar, qolganlari odatdagidek.

Agar o'rnatilmagan bo'lsa: `pip install transformers datasets scikit-learn`

In [ ]:
import os
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

## Qurilmani tanlash

BERT-base — 110 million parametrli model, shuning uchun GPU (NVIDIA uchun CUDA,
Apple Silicon uchun MPS) bu yerda oldingi darslardagidan ham muhimroq.
CPU'da ham ishlaydi, lekin bir necha barobar sekin — pastdagi giperparametrlar bo'limida
CPU uchun maslahatlar bor.

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)

## Giperparametrlar

Fine-tuning'da eng muhim raqam — **learning rate**. Noldan o'qitishda `1e-3` odatiy edi,
bu yerda esa `2e-5`, ya'ni 50 barobar kichik. Sababi: model allaqachon yaxshi og'irliklarga ega,
katta qadamlar ularni buzib yuboradi (bu hodisa *catastrophic forgetting* deyiladi).
BERT maqolasi `2e-5`, `3e-5` yoki `5e-5` ni tavsiya qiladi.

Epoxalar soni ham kichik — 2 yoki 3 tadan ko'p bo'lsa model odatda overfit bo'ladi.

Vaqtni tejash uchun 25 000 ta sharhning 5 000 tasini olamiz. **CPU'da ishlayotgan bo'lsangiz**
`TRAIN_SIZE = 1000`, `VAL_SIZE = 500`, `MAX_LEN = 128` qilib qo'ying yoki
`MODEL_NAME = "distilbert-base-uncased"` ni tanlang — u 40% kichik va 60% tezroq.

In [ ]:
MODEL_NAME = "bert-base-uncased"   # 12 layers, 768 hidden size, 110M parameters

MAX_LEN = 256          # tokens per review; anything longer is truncated
TRAIN_SIZE = 5000      # subset of the 25k training reviews
VAL_SIZE = 2000        # subset of the 25k test reviews

BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5   # fine-tuning needs a far smaller lr than training from scratch
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1     # share of steps spent warming the lr up from 0
MAX_GRAD_NORM = 1.0
LOG_EVERY = 50         # steps between progress prints

LABELS = ["salbiy", "ijobiy"]

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## IMDb dataseti

`load_dataset` to'plamni Hugging Face Hub'dan yuklab oladi va `~/.cache/huggingface` ga
saqlaydi — ikkinchi marta qayta yuklanmaydi (~80 MB).

To'plamda uchta bo'lim bor: `train` va `test` (har birida 25 000 ta belgilangan sharh)
hamda `unsupervised` (50 000 ta belgisiz sharh — biz uni ishlatmaymiz).
Belgi: `0` — salbiy, `1` — ijobiy.

In [ ]:
raw = load_dataset("imdb")
print(raw)

In [ ]:
example = raw["train"][0]

print("Label:", example["label"], "->", LABELS[example["label"]])
print("Characters:", len(example["text"]))
print()
print(example["text"][:700], "...")

## O'quv va validatsiya to'plamlari

IMDb belgilar bo'yicha **saralangan** holda keladi: avval barcha salbiy sharhlar, keyin barcha
ijobiy sharhlar. Shuning uchun `select` dan oldin albatta `shuffle` qilish kerak —
aks holda o'quv to'plamiga faqat salbiy sharhlar tushib qoladi.

Validatsiya uchun `test` bo'limidan foydalanamiz: model o'qish jarayonida ko'rmagan sharhlar.

In [ ]:
train_raw = raw["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_raw = raw["test"].shuffle(seed=SEED).select(range(VAL_SIZE))

print("Train:", len(train_raw), "| label counts:", np.bincount(train_raw["label"]))
print("Val  :", len(val_raw), "| label counts:", np.bincount(val_raw["label"]))

## Tokenizatsiya

BERT so'zlar bilan emas, **WordPiece** bo'laklari bilan ishlaydi. Lug'atda 30 522 ta element bor;
lug'atda yo'q so'z bo'laklarga bo'linadi va davomi `##` bilan belgilanadi
(masalan `unwatchable` -> `un ##wat ##cha ##ble`). Shu tufayli model hech qachon
"noma'lum so'z" bilan to'qnashmaydi.

Tokenizator uchta narsa qaytaradi:

- `input_ids` — token indekslari. Boshiga `[CLS]`, oxiriga `[SEP]` qo'shiladi.
  `[CLS]` tokenining oxirgi qatlamdagi vektori butun matnning "xulosasi" sifatida
  klassifikatsiya uchun ishlatiladi.
- `token_type_ids` — jumlalar juftligi bilan ishlaganda kerak (bizda hammasi 0).
- `attention_mask` — qaysi tokenlar haqiqiy, qaysilari padding ekanini ko'rsatadi.

Muhim: tokenizator model bilan **bir juft** bo'lishi shart, chunki indekslar aynan shu
model lug'atiga bog'langan.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = "This movie was absolutely unwatchable."
encoded = tokenizer(sample)

print("Tokens :", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("Ids    :", encoded["input_ids"])
print("Keys   :", list(encoded.keys()))
print("Mask   :", encoded["attention_mask"])
print("Vocab size:", tokenizer.vocab_size)

## Sharhlar qancha uzun?

`MAX_LEN` ni tasodifiy tanlamaslik uchun tokenlar sonining taqsimotiga qaraymiz.
Har bir qo'shimcha token hisob-kitobni sekinlashtiradi (self-attention uzunlikka nisbatan
kvadratik o'sadi), lekin juda qisqa kessak, sharhning muhim qismi yo'qoladi.

BERT umuman 512 tokendan uzun matnni qabul qila olmaydi — bu arxitekturaning qattiq cheklovi.

In [ ]:
lengths = [
    len(tokenizer(text, truncation=False, verbose=False)["input_ids"])
    for text in train_raw["text"][:1000]
]

print(f"Median: {np.median(lengths):.0f} | 90%: {np.percentile(lengths, 90):.0f} "
      f"| 95%: {np.percentile(lengths, 95):.0f} | max: {max(lengths)}")
print(f"Truncated at MAX_LEN={MAX_LEN}: {np.mean(np.array(lengths) > MAX_LEN):.1%} of reviews")

plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=60)
plt.axvline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
plt.axvline(512, color="black", linestyle=":", label="BERT limit = 512")
plt.xlabel("Tokens per review")
plt.ylabel("Reviews")
plt.title("IMDb - review length in WordPiece tokens")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Butun datasetni tokenizatsiya qilish

`map` funksiyasi tokenizatorni butun to'plamga qo'llaydi. `batched=True` bilan u bir vaqtda
1000 tadan matnni beradi — bu tokenizatorning tez Rust implementatsiyasi tufayli ancha tezroq.

Matnning o'zini olib tashlaymiz (`remove_columns`), `label` ustunini esa `labels` deb
qayta nomlaymiz — modelning `forward` metodi aynan shu nom bilan yo'qotishni o'zi hisoblab beradi.

In [ ]:
def tokenize(batch):
    # no padding here: the collator pads each batch to its own longest sequence
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)


train_ds = train_raw.map(tokenize, batched=True, remove_columns=["text"])
val_ds = val_raw.map(tokenize, batched=True, remove_columns=["text"])

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

train_ds.set_format("torch")
val_ds.set_format("torch")

print(train_ds)
print("\nFirst example ids:", train_ds[0]["input_ids"][:12].tolist())
print("First example label:", train_ds[0]["labels"].item())

## DataLoader va dinamik padding

Bir batchdagi barcha ketma-ketliklar bir xil uzunlikda bo'lishi kerak. Hammasini 256 tagacha
to'ldirish o'rniga `DataCollatorWithPadding` har bir batchni **o'sha batchdagi eng uzun**
namunagacha to'ldiradi. Qisqa sharhlar bir batchga tushsa, hisob-kitob sezilarli tejaladi.

`attention_mask` esa modelga padding tokenlarini e'tiborsiz qoldirishni aytadi.

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

batch = next(iter(train_loader))
for key, value in batch.items():
    print(f"{key:16s} {tuple(value.shape)}")

print("\nBatches per epoch:", len(train_loader))

## Modelni yuklash

`AutoModelForSequenceClassification` ikki qismdan iborat modelni yig'adi:

- **encoder** — oldindan o'qitilgan 12 qatlamli BERT (109M parametr),
- **klassifikatsiya boshi** — `[CLS]` vektorini 2 ta logitga aylantiruvchi oddiy chiziqli qatlam
  (768 x 2 = 1538 parametr), **tasodifiy** boshlang'ich qiymatlar bilan.

Shuning uchun quyida "Some weights ... are newly initialized: ['classifier.weight', ...]"
degan ogohlantirish chiqadi. Bu xato emas — aynan shu yangi qatlamni o'qitish uchun
fine-tuning qilamiz.

`id2label` va `label2id` ni berib qo'yamiz: keyin model saqlanganda bu nomlar ham saqlanadi
va bashorat `0` emas, `"ijobiy"` deb qaytadi.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "salbiy", 1: "ijobiy"},
    label2id={"salbiy": 0, "ijobiy": 1},
).to(DEVICE)

encoder_params = sum(p.numel() for p in model.base_model.parameters())
head_params = sum(p.numel() for p in model.classifier.parameters())

print(f"Pretrained encoder : {encoder_params:,} parameters")
print(f"New classifier head: {head_params:,} parameters")
print()
print(model.classifier)

## Optimizator va learning rate jadvali

Ikkita nozik jihat bor:

**Warmup.** O'qish boshida klassifikatsiya boshi butunlay tasodifiy, shuning uchun birinchi
gradientlar juda katta va ular oldindan o'qitilgan encoder'ni buzishi mumkin. Learning rate'ni
noldan boshlab sekin ko'tarish (qadamlarning 10% davomida) buni oldini oladi. Keyin u
chiziqli ravishda yana nolgacha tushadi.

**AdamW.** Oddiy Adam'dan farqi — weight decay gradientga emas, to'g'ridan-to'g'ri
og'irliklarga qo'llanadi. Transformer'larni fine-tuning qilishda standart tanlov shu.

In [ ]:
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print("Total steps:", total_steps, "| warmup steps:", warmup_steps)

## Baholash funksiyasi

Aniqlik (accuracy) bilan bir qatorda F1 ni ham hisoblaymiz. IMDb muvozanatli to'plam
bo'lgani uchun bu ikki raqam bir-biriga yaqin chiqadi, lekin nomutanosib datasetlarda
F1 ancha ishonchli ko'rsatkich.

`model(**batch)` chaqirilganda `labels` mavjud bo'lsa, model cross-entropy yo'qotishni
o'zi hisoblab, `outputs.loss` sifatida qaytaradi.

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()

    total_loss = 0.0
    preds, gold, probs = [], [], []

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)

        total_loss += outputs.loss.item() * len(batch["labels"])
        probs.append(torch.softmax(outputs.logits, dim=-1).cpu())
        preds += outputs.logits.argmax(dim=-1).cpu().tolist()
        gold += batch["labels"].cpu().tolist()

    return {
        "loss": total_loss / len(gold),
        "accuracy": accuracy_score(gold, preds),
        "f1": f1_score(gold, preds),
        "preds": np.array(preds),
        "gold": np.array(gold),
        "probs": torch.cat(probs).numpy(),
    }

## Fine-tuning'gacha bo'lgan holat

O'qishni boshlashdan oldin modelni sinab ko'ramiz. Encoder tilni biladi, lekin klassifikator
boshi tasodifiy — shuning uchun aniqlik 50% atrofida, ya'ni tanga tashlash bilan barobar
bo'lishi kutiladi. Bu bizning boshlang'ich nuqtamiz.

In [ ]:
baseline = evaluate(val_loader)

print(f"Before fine-tuning | loss: {baseline['loss']:.4f} "
      f"| accuracy: {baseline['accuracy']:.4f} | f1: {baseline['f1']:.4f}")

## Bitta epoxani o'qitish

Sikl odatdagi PyTorch sikli, ikkita qo'shimcha bilan:

- `clip_grad_norm_` — gradient normasini 1.0 bilan cheklaydi, boshlang'ich qadamlardagi
  keskin sakrashlardan himoya qiladi;
- `scheduler.step()` — har bir **qadamdan** keyin chaqiriladi (epoxadan keyin emas!),
  chunki jadval qadamlar soni bo'yicha tuzilgan.

In [ ]:
def train_one_epoch(epoch):
    model.train()

    running_loss = 0.0
    step_losses = []

    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        # gradients spike while the fresh classifier head is still random
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        step_losses.append(loss.item())

        if step % LOG_EVERY == 0:
            print(f"  epoch {epoch} | step {step:4d}/{len(train_loader)} "
                  f"| loss {running_loss / step:.4f} "
                  f"| lr {scheduler.get_last_lr()[0]:.2e}")

    return running_loss / len(train_loader), step_losses

## O'qitish

Ikki epoxa yetarli. Odatda birinchi epoxadan keyinoq validatsiya aniqligi 88-90% ga chiqadi,
ikkinchisida 90-92% ga yetadi. To'liq 25 000 ta sharhda o'qitilsa, natija ~93-94% bo'ladi.

Vaqt: GPU'da bir epoxa bir necha daqiqa, CPU'da esa ancha uzoq — sabr qiling yoki
yuqoridagi `TRAIN_SIZE` ni kamaytiring.

In [ ]:
history = {
    "val_loss": [baseline["loss"]],
    "val_acc": [baseline["accuracy"]],
    "val_f1": [baseline["f1"]],
    "train_loss": [],
}
all_step_losses = []

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, step_losses = train_one_epoch(epoch)
    metrics = evaluate(val_loader)

    all_step_losses += step_losses
    history["train_loss"].append(train_loss)
    history["val_loss"].append(metrics["loss"])
    history["val_acc"].append(metrics["accuracy"])
    history["val_f1"].append(metrics["f1"])

    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {metrics['loss']:.4f} "
        f"| Val Acc: {metrics['accuracy']:.4f} "
        f"| Val F1: {metrics['f1']:.4f}"
    )

print(f"\nTraining time: {(time.time() - start_time) / 60:.1f} min")

## Natijalar grafigi

Chapda — har bir qadamdagi yo'qotish. U juda "shovqinli" bo'ladi (batch atigi 16 ta namuna),
shuning uchun harakatlanuvchi o'rtacha qiymat ham chiziladi.

O'ngda — validatsiya aniqligi. `0` nuqtasi fine-tuning'gacha bo'lgan holat, ya'ni tasodifiy
klassifikator. Bir epoxada 50% dan 90% ga sakrash — bu oldindan o'qitilgan bilimning kuchi.

In [ ]:
window = 20
smoothed = np.convolve(all_step_losses, np.ones(window) / window, mode="valid")
epochs_axis = range(0, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(all_step_losses, alpha=0.3, label="Step loss")
axes[0].plot(range(window - 1, len(all_step_losses)), smoothed, label=f"Moving avg ({window})")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Cross Entropy Loss")
axes[0].set_title("Training loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_axis, history["val_acc"], marker="o", label="Validation accuracy")
axes[1].plot(epochs_axis, history["val_f1"], marker="s", label="Validation F1")
axes[1].axhline(0.5, color="gray", linestyle="--", label="Random guess")
axes[1].set_xlabel("Epoch (0 = before fine-tuning)")
axes[1].set_ylabel("Score")
axes[1].set_title("BERT-base - IMDb")
axes[1].set_xticks(list(epochs_axis))
axes[1].set_ylim(0.4, 1.0)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Chalkashlik matritsasi

Aniqlik bitta raqam, u qanday xatolar borligini ko'rsatmaydi. Chalkashlik matritsasi
to'rt kataklikni beradi: diagonal — to'g'ri bashoratlar, diagonaldan tashqarisi — xatolar.

Agar bitta ustun boshqasidan sezilarli "og'ir" bo'lsa, model bir tomonga qiyshaygan degani.

In [ ]:
final = evaluate(val_loader)
cm = confusion_matrix(final["gold"], final["preds"])

fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(cm, cmap="Blues")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)

ax.set_xticks([0, 1], labels=LABELS)
ax.set_yticks([0, 1], labels=LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix - accuracy {final['accuracy']:.2%}")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## Model qayerda xato qildi?

Xato bashoratlarni o'qish — modelni tushunishning eng foydali usuli. IMDb'da xatolar odatda
uch turdagi sharhlarda uchraydi: kinoyali ("ajoyib, yana bir shedevr..."), aralash fikrli
("o'yin yaxshi, lekin syujet dahshat") va MAX_LEN tufayli kesilgan uzun sharhlarda —
sharhning xulosasi oxirida bo'lsa, model uni umuman ko'rmaydi.

In [ ]:
wrong = np.where(final["preds"] != final["gold"])[0]
print(f"Wrong predictions: {len(wrong)} / {len(final['gold'])}")

for idx in wrong[:3]:
    idx = int(idx)
    confidence = final["probs"][idx].max()
    print(f"\n--- true: {LABELS[final['gold'][idx]]} "
          f"| predicted: {LABELS[final['preds'][idx]]} ({confidence:.1%}) ---")
    print(val_raw[idx]["text"][:450], "...")

## O'z matningizni sinash

Model ingliz tilidagi film sharhlarida o'qitilgan, shuning uchun sinov matnlari ham
ingliz tilida bo'lgani ma'qul. Softmax ehtimolliklari modelning qanchalik ishonchli
ekanini ko'rsatadi.

O'zbek tilidagi matnlar uchun ko'p tilli asos kerak bo'ladi: `bert-base-multilingual-cased`
yoki `xlm-roberta-base` — kod esa aynan shunday qoladi, faqat `MODEL_NAME` o'zgaradi.

In [ ]:
@torch.no_grad()
def predict(texts):
    model.eval()

    encoded = tokenizer(
        texts, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt"
    ).to(DEVICE)
    probs = torch.softmax(model(**encoded).logits, dim=-1).cpu()

    for text, prob in zip(texts, probs):
        label = LABELS[int(prob.argmax())]
        print(f"{label:8s} {prob.max():.1%}  <- {text}")


predict([
    "One of the best films I have seen this year, the acting was superb.",
    "I want my two hours back. Boring, predictable and badly acted.",
    "It started slow, but the ending completely won me over.",
    "Great. Another sequel nobody asked for.",
])

## Modelni saqlash va qayta yuklash

`save_pretrained` og'irliklarni (`model.safetensors`), konfiguratsiyani (`config.json`,
belgi nomlari bilan birga) va tokenizator fayllarini saqlaydi. Shu papkani
`from_pretrained` ga berib, modelni istalgan joyda qayta tiklash mumkin.

`pipeline` esa tokenizatsiya, model va softmax'ni bitta chaqiruvga jamlaydi — ishlab
chiqarishda eng qulay usul.

> Papka ~440 MB joy egallaydi. Git bilan ishlayotgan bo'lsangiz, uni `.gitignore` ga
> qo'shib qo'ying.

In [ ]:
SAVE_DIR = "bert_imdb_sentiment"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(sorted(os.listdir(SAVE_DIR)))

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR)

print(classifier("A charming, funny and surprisingly moving little film."))
print(classifier("The plot made no sense and I fell asleep halfway through."))

## Xulosa

Nima qildik:

1. IMDb sharhlarini WordPiece tokenizatori bilan BERT tushunadigan ko'rinishga keltirdik.
2. Oldindan o'qitilgan encoder tepasiga yangi klassifikatsiya boshi qo'ydik.
3. Butun tarmoqni kichik learning rate, warmup va gradient clipping bilan 2 epoxa o'qitdik.
4. 50% (tasodifiy) dan ~90%+ aniqlikka chiqdik — atigi 5 000 ta namunada.

Asosiy xulosa: **fine-tuning noldan o'qitishdan minglab marta arzon**. word2vec darsida
yaxshi vektorlar uchun 100 MB matn va bir necha epoxa kerak bo'lgan edi; bu yerda esa
tayyor bilimning ustiga bir necha daqiqada ishlaydigan klassifikator qurdik.

Keyingi qadamlar:

- `TRAIN_SIZE = 25000` qilib to'liq datasetda o'qiting — aniqlik ~93-94% ga chiqadi.
- `distilbert-base-uncased` bilan solishtiring: 2 barobar tez, aniqlik ~1% pastroq.
- `roberta-base` ni sinab ko'ring — u NSP'siz, ko'proq ma'lumotda o'qitilgan va odatda
  BERT'dan yaxshiroq natija beradi.
- Encoder'ning quyi qatlamlarini muzlatib (`param.requires_grad = False`) tezlik va
  aniqlik o'rtasidagi muvozanatni tekshiring.
- Uch sinfli (ijobiy / neytral / salbiy) vazifaga o'ting: `num_labels=3` va boshqa dataset,
  masalan `tweet_eval` yoki `sst5`.